In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def corr2d(X,K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0]-h+1, X.shape[1]-w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = (X[i:i+h, j:j +w]*K).sum() # elementwise mul
    return Y            

In [3]:
class Conv2d(nn.Module): # conv layer
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self,x):
        return corr2d(x,self.weight) + self.bias    

In [4]:
# create an image
X =  torch.ones((6,8))
X[:,2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [5]:
K = torch.tensor([[1.0,-1.0]])
Y=corr2d    (X,K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [6]:
corr2d(X.t(),K) # transposed image doesnt work here since it detects only vertical edges

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

In [7]:
conv2d = nn.LazyConv2d(1,kernel_size=(1,2), bias = False)
X =X.reshape((1,1,6,8))
Y =Y.reshape((1,1,6,7))
lr = 3e-2

for i in range(10):
    Y_hat =conv2d(X)
    l = (Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()

    conv2d.weight.data[:] -=lr*conv2d.weight.grad
    if(i+1)%2==0:
        print(f"epoch {i+1}, loss {l.sum():.3f}")


epoch 2, loss 12.074
epoch 4, loss 3.227
epoch 6, loss 1.033
epoch 8, loss 0.375
epoch 10, loss 0.145


In [8]:
def comp_conv2d(conv2d, X):
    X = X.reshape((1,1)+X.shape)
    Y=conv2d(X) # requires 4d so we add 1,1 to 8,8
    return Y.reshape(Y.shape[2:]) # remove extra added dims

conv2d = nn.LazyConv2d(1,kernel_size=3,padding = 1)
X=torch.rand(size =(8,8))
comp_conv2d(conv2d, X)



tensor([[ 0.5320, -0.0006,  0.2229,  0.1897,  0.4163,  0.4918,  0.1136,  0.1061],
        [ 0.5536,  0.2754,  0.2592,  0.0671, -0.0363,  0.2581,  0.2386,  0.2341],
        [ 0.3351,  0.0491,  0.4563,  0.3495, -0.0085,  0.0025,  0.3038,  0.3335],
        [ 0.4992,  0.0813, -0.0599,  0.1704,  0.2326,  0.2777,  0.3432,  0.3441],
        [ 0.3685,  0.3774,  0.1851,  0.2780,  0.2245, -0.0059, -0.0301,  0.4211],
        [ 0.1884,  0.3532,  0.2580,  0.2220,  0.1295,  0.2321,  0.1349,  0.2838],
        [ 0.2256,  0.1144,  0.1593,  0.1251,  0.3104,  0.1691,  0.1636,  0.2719],
        [ 0.2658,  0.0565,  0.2277,  0.1826,  0.3186,  0.1671,  0.1257,  0.3318]],
       grad_fn=<ViewBackward0>)

In [9]:
def corr2d_multi_in(X,K):
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K))

In [10]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)


tensor([[ 56.,  72.],
        [104., 120.]])

In [11]:
def corr2d_multi_in_out(X,K):
    return torch.stack([corr2d_multi_in(X,k) for k in K], 0)

In [12]:
def corr2d_multi_in_out_1x1(X,K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X.shape = X.reshape((c_i,h*w))
    K.shape = K.reshape((c_o,c_i))
    Y = torch.matmul(X,K)
    return Y.reshape((c_o, h, w))

In [13]:
def pool2d(X,pool_size, mode = 'max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] -p_h+1,X.shape[1]-p_w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i,j] = X[i:i+p_h, j:j+p_w].max()
            elif mode =='avg':    
                Y[i, j] = X[i: i + p_h, j: j + p_w].mean()

In [16]:
def init_cnn(module):
    if type(module)==nn.linear or type(module) ==nn.Conv2d:
        nn.init.xavier_uniform_(module.weight)

class LeNet(d2l.Classifier):
    def __init__(self, lr =0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(

            nn.LazyConv2d(6, kernel_size=5, padding=2), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.LazyConv2d(16, kernel_size=5), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.LazyLinear(120), nn.Sigmoid(),
            nn.LazyLinear(84), nn.Sigmoid(),
            nn.LazyLinear(num_classes))
        
